### Imports

In [ ]:
import os
import pandas as pd
import numpy as np
import re
from astropy.io import fits
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (roc_auc_score, accuracy_score, precision_score, recall_score, f1_score,confusion_matrix, classification_report)
from imblearn.ensemble import BalancedRandomForestClassifier

### Paths

In [ ]:
SEED = 67
table_multi_path = r"C:\Users\anacs\OneDrive\Área de Trabalho\ic-iag\Work 2\ALL_MorphoSPLUS_GalfitM_output_splus.csv"
labels_table_path = r"C:/Users/anacs/OneDrive\Área de Trabalho\ic-iag\Work\tabela_filtrada.csv"
images_path = r"C:/Users/anacs/OneDrive/Área de Trabalho/ic-iag/Work 2/Imagens_CG_2"


### Prepare data

In [ ]:
#Read and merge
df_tab = pd.read_csv(table_multi_path, low_memory=False)
df_labels = pd.read_csv(labels_table_path, low_memory=False)

df_tab["ID_1"] = df_tab["ID_1"].astype(str)
df_labels["ID"] = df_labels["ID"].astype(str)

# 0 = good, 1 = bad
df_labels["label"] = df_labels["type"].apply(lambda x: 0 if int(x) == 0 else 1).astype(int)

df = df_tab.merge(
    df_labels[["ID", "label"]],
    left_on="ID_1",
    right_on="ID",
    how="inner"
).drop(columns=["ID"])

df["label"] = df["label"].astype(int)

print("After merge:", df.shape)
print(df["label"].value_counts())


#Convert to numeric
text_cols = ["source_folder", "source_file", "ID_1", "Field_ID", "Dir_", "Field"]

def limpar_para_numero(serie):
    serie = serie.astype(str)
    serie = serie.str.replace("*", "", regex=False)
    serie = serie.str.replace(",", ".", regex=False)
    serie = serie.str.strip()
    serie = serie.replace({"nan": np.nan, "None": np.nan, "": np.nan})
    return pd.to_numeric(serie, errors="coerce")

for col in df.columns:
    if col in text_cols or col == "label":
        continue
    df[col] = limpar_para_numero(df[col])

#All features
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols.remove("label")

features_all = []
for c in numeric_cols:
    if df[c].notna().sum() < 30:
        continue
    if df[c].nunique(dropna=True) < 2:
        continue
    features_all.append(c)

print("\nAll features:", len(features_all))

#Selected features
y = df["label"].values

resultados = []
MIN_NON_NAN = 200

for col in features_all:
    mask = df[col].notna().values
    if mask.sum() < MIN_NON_NAN:
        continue

    x_sub = df.loc[mask, col].values
    y_sub = y[mask]

    if len(np.unique(y_sub)) < 2:
        continue

    try:
        auc = roc_auc_score(y_sub, x_sub)
        sep = max(auc, 1 - auc)
        resultados.append([col, sep])
    except:
        pass

df_auc = pd.DataFrame(resultados, columns=["feature", "auc_sep"])
df_auc = df_auc.sort_values("auc_sep", ascending=False).reset_index(drop=True)

AUC_THRESHOLD = 0.60
df_good = df_auc[df_auc["auc_sep"] > AUC_THRESHOLD].copy().reset_index(drop=True)

features_boas = df_good["feature"].tolist()
auc_dict = dict(zip(df_good["feature"], df_good["auc_sep"]))

CORR_THRESHOLD = 0.90
remover = set()

if len(features_boas) > 1:
    corr_abs = df[features_boas].corr(method="pearson", min_periods=200).abs()

    for i in range(len(features_boas)):
        for j in range(i + 1, len(features_boas)):
            f1 = features_boas[i]
            f2 = features_boas[j]

            if f1 in remover or f2 in remover:
                continue

            val = corr_abs.loc[f1, f2]
            if np.isfinite(val) and val >= CORR_THRESHOLD:
                if auc_dict[f1] >= auc_dict[f2]:
                    remover.add(f2)
                else:
                    remover.add(f1)

features_sel = [f for f in features_boas if f not in remover]

print("Selected features:", len(features_sel))


### Split and function

In [ ]:
#Split
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=SEED, stratify=df["label"]
)

y_train = train_df["label"].values
y_test = test_df["label"].values


#Plots
def plot_confusion(cm, titulo):
    cm_perc = cm / cm.sum(axis=1, keepdims=True)

    plt.figure(figsize=(5.5, 4.8))
    plt.imshow(cm_perc, cmap="Blues", vmin=0, vmax=1)
    plt.title(titulo)
    plt.xticks([0, 1], ["Good", "Bad"])
    plt.yticks([0, 1], ["Good", "Bad"])
    plt.xlabel("Predicted")
    plt.ylabel("True")

    for i in range(2):
        for j in range(2):
            plt.text(
                j, i,
                f"{cm[i, j]}\n({cm_perc[i, j]*100:.1f}%)",
                ha="center", va="center", color="black"
            )

    plt.tight_layout()
    plt.show()


FNAME_RE = re.compile(
    r"^MorphoSPLUS_\d+_(DR4_3_STRIPE82-\d{4}_\d{7})_(real|model|resid)_([a-z0-9]+)\.fits$",
    re.IGNORECASE
)

def plot_misclassified_fits(nome, test_df, y_true, y_pred, images_path, max_images=6):
    fits_index = {}

    for fname in os.listdir(images_path):
        m = FNAME_RE.match(fname)
        if not m:
            continue

        object_id = m.group(1)
        kind = m.group(2).lower()
        flt = m.group(3).lower()

        key = (object_id, flt)

        if key not in fits_index:
            fits_index[key] = {}

        fits_index[key][kind] = os.path.join(images_path, fname)

    mis_idx = np.where(y_true != y_pred)[0]

    if len(mis_idx) == 0:
        print(f"[{nome}] Nenhuma misclassified.")
        return {}

    rng = np.random.default_rng(SEED)
    mis_idx = rng.permutation(mis_idx)

    plt.figure(figsize=(12, 2.8 * max_images))
    plotted = 0

    used_filter = {}  # <-- NOVO

    for idx in mis_idx:
        if plotted >= max_images:
            break

        row = test_df.iloc[idx]
        obj_id = str(row["ID_1"]).strip().rstrip(".")

        filtros_disponiveis = []
        for (oid, flt) in fits_index.keys():
            if oid == obj_id:
                filtros_disponiveis.append(flt)

        if len(filtros_disponiveis) == 0:
            continue

        flt = filtros_disponiveis[0]
        files = fits_index[(obj_id, flt)]

        if "resid" not in files or "model" not in files or "real" not in files:
            continue

        with fits.open(files["resid"]) as hdul:
            resid = np.squeeze(hdul[0].data.astype(np.float32))
        with fits.open(files["model"]) as hdul:
            model_img = np.squeeze(hdul[0].data.astype(np.float32))
        with fits.open(files["real"]) as hdul:
            real = np.squeeze(hdul[0].data.astype(np.float32))

        true_lab = int(y_true[idx])
        pred_lab = int(y_pred[idx])

        titulo = f"{obj_id} | {flt} | true={true_lab} pred={pred_lab}"

        r = plotted

        plt.subplot(max_images, 3, r * 3 + 1)
        plt.imshow(resid, cmap="gray", origin="lower")
        plt.title(titulo + "\nresid", fontsize=10)
        plt.axis("off")

        plt.subplot(max_images, 3, r * 3 + 2)
        plt.imshow(model_img, cmap="gray", origin="lower")
        plt.title("model", fontsize=10)
        plt.axis("off")

        plt.subplot(max_images, 3, r * 3 + 3)
        plt.imshow(real, cmap="gray", origin="lower")
        plt.title("real", fontsize=10)
        plt.axis("off")

        used_filter[obj_id] = flt
        plotted += 1

    plt.tight_layout()
    plt.show()

    return used_filter


#save
flt_re = re.compile(r"_([a-z0-9]+)(?:_aug_\d+)?$", re.IGNORECASE)

def extract_filter_from_filename(fname: str) -> str:
    base = str(fname).replace(".fits", "")
    m = flt_re.search(base)
    return m.group(1).lower() if m else "UNKNOWN"

#Function
def model(nome, modelo, features):
    X_train = train_df[features]
    X_test = test_df[features]

    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", modelo)
    ])

    pipe.fit(X_train, y_train)

    prob = pipe.predict_proba(X_test)[:, 1]
    pred = (prob >= 0.25).astype(int)

    auc = roc_auc_score(y_test, prob)
    acc = accuracy_score(y_test, pred)
    prec = precision_score(y_test, pred, zero_division=0)
    rec = recall_score(y_test, pred, zero_division=0)
    f1 = f1_score(y_test, pred, zero_division=0)

    print(f"\n{nome}")
    print("N features:", len(features))
    print("AUC:", auc)
    print("Accuracy:", acc)
    print("Precision:", prec)
    print("Recall:", rec)
    print("F1:", f1)
    print("\nClassification report:")
    print(classification_report(y_test, pred, target_names=["Good (0)", "Bad (1)"]))

    cm = confusion_matrix(y_test, pred, labels=[0, 1])
    plot_confusion(cm, f"Confusion Matrix - {nome}")

    used_filter = plot_misclassified_fits(nome, test_df, y_test, pred, images_path, max_images=6)

    object_ids = test_df["ID_1"].astype(str).str.strip().str.rstrip(".").values

    filt_list = []
    for oid in object_ids:
        filt_list.append(used_filter.get(oid, "UNKNOWN"))

    df_rf = pd.DataFrame({
        "object_id": object_ids,
        "filter": np.array(filt_list, dtype=str),
        "filename": np.array([""] * len(object_ids), dtype=str),  # RF não tem filename FITS
        "y_true": y_test.astype(int),
        "y_pred": pred.astype(int),
    })

    df_rf["mis"] = df_rf["y_true"] != df_rf["y_pred"]
    df_rf["key"] = list(zip(df_rf["object_id"], df_rf["filter"]))
    df_rf["model"] = nome

    return df_rf



### Main

In [ ]:
rf_balanced = BalancedRandomForestClassifier(
    n_estimators=1000,
    random_state=SEED,
    n_jobs=-1,
    min_samples_leaf=2
)

df_rf_all = model("RF_ALL", rf_balanced, features_all)
df_rf_sel = model("RF_SELECTED", rf_balanced, features_sel)
